
Design an automated pipeline that runs at scheduled intervals (e.g., every 12 hours or daily) to:

1. Collect new reviews from the Google Play Store.

2. Preprocess the data (clean, transform, and validate).

3. Store the new data into the SQLite database.

1. Data Collection
   Use the google_play_scraper to fetch new reviews.
   Keep track of the last collected review to avoid duplicates.
   However, note that the scraper returns reviews sorted by time, and we are getting the newest ones.
   We can use the at (timestamp) field to determine the last collected review.

# 🤖 Automation Pipeline Design

In [1]:
import os
from datetime import timedelta
import time
import logging
from google_play_scraper import reviews, Sort
import pandas as pd
from datetime import datetime
import sqlite3
import logging
from sqlalchemy import create_engine, text
import schedule
import time
import logging
from datetime import datetime
import os
import sys
import sqlite3
from sqlalchemy import create_engine, text

#### Configuration Management (config/settings.py)

In [2]:
class Config:
    # 数据库配置
    DATABASE_URL = "sqlite:///chatgpt_reviews.db"
    
    # 收集配置
    APP_ID = "com.openai.chatgpt"
    LANGUAGE = "en"
    COUNTRY = "us"
    MAX_REVIEWS_PER_RUN = 1000
    
    # 调度配置
    COLLECTION_INTERVAL_HOURS = 12
    
    # 文件路径
    LOG_DIR = "logs"
    DATA_DIR = "data"

#### Data Collection Module (src/data_collection.py)

In [3]:
class ReviewCollector:
    def __init__(self):
        self.setup_logging()
        
    def setup_logging(self):
        logging.basicConfig(
            level=logging.INFO,
            format='%(asctime)s - %(levelname)s - %(message)s',
            handlers=[
                logging.FileHandler('logs/collection.log'),
                logging.StreamHandler()
            ]
        )
        self.logger = logging.getLogger(__name__)
    
    def collect_new_reviews(self, continuation_token=None):
        """
        Collect new reviews with error handling and rate limiting
        """
        try:
            self.logger.info("Starting review collection...")
            
            # Collect reviews with pagination support
            result, continuation_token = reviews(
                Config.APP_ID,
                lang=Config.LANGUAGE,
                country=Config.COUNTRY,
                sort=Sort.NEWEST,
                count=Config.MAX_REVIEWS_PER_RUN,
                continuation_token=continuation_token
            )
            
            self.logger.info(f"Collected {len(result)} new reviews")
            return result, continuation_token
            
        except Exception as e:
            self.logger.error(f"Collection failed: {e}")
            return [], None
    
    def validate_review_data(self, review):
        """
        Validate individual review data quality
        """
        if not review.get('reviewId'):
            return False
            
        if not review.get('content') or len(review['content']) < Config.MIN_REVIEW_LENGTH:
            return False
            
        if len(review['content']) > Config.MAX_REVIEW_LENGTH:
            return False
            
        if not isinstance(review.get('score'), int) or not (1 <= review['score'] <= 5):
            return False
            
        return True

#### Data Processing Module (src/data_processing.py)


In [4]:
import pandas as pd
import logging

class DataProcessor:
    def __init__(self):
        self.logger = logging.getLogger(__name__)
    
    def process_raw_reviews(self, raw_reviews):
        """
        简化的数据处理 - 只做必要的清理
        """
        if not raw_reviews:
            return pd.DataFrame()
        
        df = pd.DataFrame(raw_reviews)
        
        # 基本清理
        df_clean = self.clean_review_data(df)
        
        self.logger.info(f"Processed {len(df_clean)} reviews")
        return df_clean
    
    def clean_review_data(self, df):
        """
        只做必要的清理操作
        """
        # 删除不必要的列
        columns_to_drop = ['userImage', 'replyContent', 'repliedAt']
        existing_columns_to_drop = [col for col in columns_to_drop if col in df.columns]
        df = df.drop(columns=existing_columns_to_drop)
        
        # 处理缺失值
        df['reviewCreatedVersion'] = df['reviewCreatedVersion'].fillna('unknown')
        df['appVersion'] = df['appVersion'].fillna('unknown')
        
        # 确保时间格式正确
        df['at'] = pd.to_datetime(df['at']).dt.strftime('%Y-%m-%d %H:%M:%S')
        
        return df

#### Database Manager (src/database_manager.py)

In [5]:
class DatabaseManager:
    def __init__(self):
        self.engine = create_engine(Config.DATABASE_URL)
        self.logger = logging.getLogger(__name__)
        self.setup_database()
    
    def setup_database(self):
        """
        简化的数据库设置 - 只使用原始表结构
        """
        try:
            with self.engine.connect() as conn:
                # 创建表（使用原始结构）
                create_table_query = """
                CREATE TABLE IF NOT EXISTS reviews (
                    reviewId TEXT PRIMARY KEY,
                    userName TEXT NOT NULL,
                    content TEXT,
                    score INTEGER NOT NULL,
                    thumbsUpCount INTEGER DEFAULT 0,
                    reviewCreatedVersion TEXT,
                    at TEXT NOT NULL,
                    appVersion TEXT
                )
                """
                conn.execute(text(create_table_query))
                
                # 只创建核心索引
                indexes = [
                    "CREATE INDEX IF NOT EXISTS idx_score ON reviews(score)",
                    "CREATE INDEX IF NOT EXISTS idx_app_version ON reviews(appVersion)",
                    "CREATE INDEX IF NOT EXISTS idx_thumbs_up ON reviews(thumbsUpCount)"
                ]
                
                for index_query in indexes:
                    conn.execute(text(index_query))
                
                conn.commit()
                self.logger.info("Database setup completed")
                
        except Exception as e:
            self.logger.error(f"Database setup failed: {e}")
            raise
    
    def insert_new_reviews(self, df):
        """
        插入新评论，自动去重
        """
        if df.empty:
            return 0
        
        try:
            # 检查现有评论
            existing_reviews = pd.read_sql(
                "SELECT reviewId FROM reviews", 
                self.engine
            )
            
            # 过滤掉重复项
            new_reviews = df[~df['reviewId'].isin(existing_reviews['reviewId'])]
            
            if new_reviews.empty:
                self.logger.info("No new reviews to insert")
                return 0
            
            # 只插入原始列
            columns_to_insert = [
                'reviewId', 'userName', 'content', 'score', 
                'thumbsUpCount', 'reviewCreatedVersion', 'at', 'appVersion'
            ]
            
            new_reviews[columns_to_insert].to_sql(
                'reviews', 
                self.engine, 
                if_exists='append', 
                index=False
            )
            
            self.logger.info(f"Inserted {len(new_reviews)} new reviews")
            return len(new_reviews)
            
        except Exception as e:
            self.logger.error(f"Insert failed: {e}")
            return 0
    
    def get_database_stats(self):
        """
        获取数据库统计信息
        """
        try:
            stats = pd.read_sql("""
                SELECT 
                    COUNT(*) as total_reviews,
                    AVG(score) as avg_score,
                    MAX(at) as latest_review
                FROM reviews
            """, self.engine)
            
            return {
                'total_reviews': stats['total_reviews'][0],
                'avg_score': round(stats['avg_score'][0], 2),
                'latest_review': stats['latest_review'][0]
            }
            
        except Exception as e:
            self.logger.error(f"Stats query failed: {e}")
            return {}

#### Scheduler (automation.py)

In [6]:
import pandas as pd
import sqlite3
import schedule
import time
import logging
import os
import sys
import traceback
from sqlalchemy import create_engine, text
from google_play_scraper import reviews, Sort
from datetime import datetime

# 配置
class Config:
    DATABASE_URL = "sqlite:///chatgpt_reviews.db"
    APP_ID = "com.openai.chatgpt"
    LANGUAGE = "en"
    COUNTRY = "us"
    MAX_REVIEWS_PER_RUN = 100  # 改为较小值测试
    COLLECTION_INTERVAL_HOURS = 6  # 改为更频繁

class ChatGPTReviewAutomation:
    def __init__(self):
        self.engine = create_engine(Config.DATABASE_URL)
        self.continuation_token = None
        self.setup_logging()
        self.setup_database()
    
    def setup_logging(self):
        """设置日志"""
        os.makedirs('logs', exist_ok=True)
        logging.basicConfig(
            level=logging.INFO,
            format='%(asctime)s - %(levelname)s - %(message)s',
            handlers=[
                logging.FileHandler('logs/collection.log'),
                logging.StreamHandler()
            ]
        )
        self.logger = logging.getLogger(__name__)
    
    def setup_database(self):
        """设置数据库"""
        try:
            with self.engine.connect() as conn:
                # 创建表
                create_table_query = """
                CREATE TABLE IF NOT EXISTS reviews (
                    reviewId TEXT PRIMARY KEY,
                    userName TEXT NOT NULL,
                    content TEXT,
                    score INTEGER NOT NULL,
                    thumbsUpCount INTEGER DEFAULT 0,
                    reviewCreatedVersion TEXT,
                    at TEXT NOT NULL,
                    appVersion TEXT
                )
                """
                conn.execute(text(create_table_query))
                
                # 创建索引
                indexes = [
                    "CREATE INDEX IF NOT EXISTS idx_score ON reviews(score)",
                    "CREATE INDEX IF NOT EXISTS idx_app_version ON reviews(appVersion)",
                    "CREATE INDEX IF NOT EXISTS idx_thumbs_up ON reviews(thumbsUpCount)"
                ]
                
                for index_query in indexes:
                    conn.execute(text(index_query))
                
                conn.commit()
                self.logger.info("Database setup completed")
                
        except Exception as e:
            self.logger.error(f"Database setup failed: {e}")
            raise
    
    def collect_reviews(self):
        """收集评论 - 修复版本"""
        try:
            self.logger.info("Starting review collection...")
            self.logger.info(f"Config: APP_ID={Config.APP_ID}, LANG={Config.LANGUAGE}, COUNTRY={Config.COUNTRY}")
            
            # 重要修复：每次重置 continuation_token 来获取最新评论
            self.continuation_token = None
            
            # 移除了 timeout 参数，因为 google-play-scraper 不支持
            result, self.continuation_token = reviews(
                Config.APP_ID,
                lang=Config.LANGUAGE,
                country=Config.COUNTRY,
                sort=Sort.NEWEST,
                count=Config.MAX_REVIEWS_PER_RUN,
                continuation_token=self.continuation_token
            )
            
            self.logger.info(f"Collected {len(result)} new reviews")
            
            # 添加详细日志来调试
            if result:
                sample = result[0]
                self.logger.info(f"Sample review - User: {sample.get('userName', 'N/A')}, Score: {sample.get('score', 'N/A')}, Date: {sample.get('at', 'N/A')}")
                self.logger.info(f"Sample content: {sample.get('content', 'N/A')[:100]}...")
            else:
                self.logger.warning("No reviews collected in this batch")
                
            return result
            
        except Exception as e:
            self.logger.error(f"Collection failed: {e}")
            # 添加完整错误跟踪
            self.logger.error(f"Full error traceback: {traceback.format_exc()}")
            return []
    
    def process_reviews(self, raw_reviews):
        """处理评论"""
        if not raw_reviews:
            self.logger.warning("No raw reviews to process")
            return pd.DataFrame()
        
        df = pd.DataFrame(raw_reviews)
        self.logger.info(f"Processing {len(df)} raw reviews")
        
        # 清理数据
        columns_to_drop = ['userImage', 'replyContent', 'repliedAt']
        existing_columns_to_drop = [col for col in columns_to_drop if col in df.columns]
        df = df.drop(columns=existing_columns_to_drop)
        
        df['reviewCreatedVersion'] = df['reviewCreatedVersion'].fillna('unknown')
        df['appVersion'] = df['appVersion'].fillna('unknown')
        df['at'] = pd.to_datetime(df['at']).dt.strftime('%Y-%m-%d %H:%M:%S')
        
        self.logger.info(f"Processed {len(df)} reviews")
        return df
    
    def insert_reviews(self, df):
        """插入评论"""
        if df.empty:
            self.logger.warning("DataFrame is empty, nothing to insert")
            return 0
        
        try:
            # 检查现有评论
            existing_reviews = pd.read_sql("SELECT reviewId FROM reviews", self.engine)
            self.logger.info(f"Found {len(existing_reviews)} existing reviews in database")
            
            # 过滤重复
            new_reviews = df[~df['reviewId'].isin(existing_reviews['reviewId'])]
            self.logger.info(f"After deduplication: {len(new_reviews)} new reviews")
            
            if new_reviews.empty:
                self.logger.info("No new reviews to insert (all duplicates)")
                return 0
            
            # 插入新评论
            new_reviews.to_sql('reviews', self.engine, if_exists='append', index=False)
            
            self.logger.info(f"Inserted {len(new_reviews)} new reviews")
            return len(new_reviews)
            
        except Exception as e:
            self.logger.error(f"Insert failed: {e}")
            self.logger.error(f"Full traceback: {traceback.format_exc()}")
            return 0
    
    def get_stats(self):
        """获取统计信息"""
        try:
            stats = pd.read_sql("""
                SELECT 
                    COUNT(*) as total_reviews,
                    AVG(score) as avg_score,
                    MAX(at) as latest_review
                FROM reviews
            """, self.engine)
            
            result = {
                'total_reviews': stats['total_reviews'][0],
                'avg_score': round(stats['avg_score'][0], 2) if stats['avg_score'][0] else 0,
                'latest_review': stats['latest_review'][0]
            }
            self.logger.info(f"Database stats: {result}")
            return result
            
        except Exception as e:
            self.logger.error(f"Stats query failed: {e}")
            return {}
    
    def run_job(self):
        """运行收集任务"""
        self.logger.info("=== Starting collection job ===")
        
        try:
            # 收集数据
            raw_reviews = self.collect_reviews()
            
            # 处理数据
            processed_reviews = self.process_reviews(raw_reviews)
            
            # 存储数据
            if not processed_reviews.empty:
                new_count = self.insert_reviews(processed_reviews)
                
                # 获取统计
                stats = self.get_stats()
                self.logger.info(f"Job completed. New: {new_count}, Total: {stats.get('total_reviews', 0)}")
            else:
                self.logger.info("No new reviews to process")
                
        except Exception as e:
            self.logger.error(f"Collection job failed: {e}")
            self.logger.error(f"Full traceback: {traceback.format_exc()}")
    
    def start(self):
        """启动自动化"""
        self.logger.info(f"Starting scheduler with {Config.COLLECTION_INTERVAL_HOURS}-hour intervals")
        
        # 安排任务
        schedule.every(Config.COLLECTION_INTERVAL_HOURS).hours.do(self.run_job)
        
        # 立即运行一次
        self.run_job()
        
        # 保持运行
        while True:
            try:
                schedule.run_pending()
                time.sleep(60)
            except KeyboardInterrupt:
                self.logger.info("Scheduler stopped by user")
                break
            except Exception as e:
                self.logger.error(f"Scheduler error: {e}")
                time.sleep(300)

def main():
    print("🚀 Starting ChatGPT Reviews Automation Pipeline")
    print("Database: SQLite3")
    print("Collection Interval: 6 hours")
    print("Press Ctrl+C to stop the scheduler")
    
    try:
        automation = ChatGPTReviewAutomation()
        automation.start()
    except KeyboardInterrupt:
        print("\n👋 Automation pipeline stopped")
    except Exception as e:
        print(f"❌ Fatal error: {e}")
        sys.exit(1)

if __name__ == "__main__":
    main()

2025-11-16 11:30:53,467 - INFO - Database setup completed
2025-11-16 11:30:53,467 - INFO - Starting scheduler with 6-hour intervals
2025-11-16 11:30:53,468 - INFO - === Starting collection job ===
2025-11-16 11:30:53,468 - INFO - Starting review collection...
2025-11-16 11:30:53,468 - INFO - Config: APP_ID=com.openai.chatgpt, LANG=en, COUNTRY=us


🚀 Starting ChatGPT Reviews Automation Pipeline
Database: SQLite3
Collection Interval: 6 hours
Press Ctrl+C to stop the scheduler


2025-11-16 11:34:38,559 - INFO - Collected 0 new reviews
2025-11-16 11:34:38,568 - WARNING - No reviews collected in this batch
2025-11-16 11:34:38,569 - WARNING - No raw reviews to process
2025-11-16 11:34:38,592 - INFO - No new reviews to process
2025-11-16 11:40:08,688 - INFO - Scheduler stopped by user


In [7]:
import subprocess
import socket
import requests
import os
import time

def deep_network_diagnosis():
    print("=== 深度网络诊断 ===")
    
    # 1. 检查路由跟踪
    print("\n1. 路由跟踪到 Google...")
    try:
        result = subprocess.run(["traceroute", "-m", "10", "www.google.com"], 
                              capture_output=True, text=True, timeout=30)
        print("路由跟踪结果:")
        print(result.stdout[:500])  # 只显示前500字符
    except Exception as e:
        print(f"路由跟踪失败: {e}")
    
    # 2. 检查网络接口
    print("\n2. 网络接口状态...")
    try:
        result = subprocess.run(["ifconfig"], capture_output=True, text=True)
        interfaces = [line for line in result.stdout.split('\n') if 'status:' in line]
        for interface in interfaces:
            print(interface)
    except Exception as e:
        print(f"接口检查失败: {e}")
    
    # 3. 检查系统代理
    print("\n3. 系统代理设置...")
    try:
        # 检查网络设置
        networks = subprocess.run(["networksetup", "-listallnetworkservices"], 
                                capture_output=True, text=True).stdout.split('\n')[1:]
        
        for network in networks:
            if network.strip():
                http_proxy = subprocess.run(["networksetup", "-getwebproxy", network.strip()], 
                                          capture_output=True, text=True)
                https_proxy = subprocess.run(["networksetup", "-getsecurewebproxy", network.strip()], 
                                           capture_output=True, text=True)
                
                if "Yes" in http_proxy.stdout or "Yes" in https_proxy.stdout:
                    print(f"⚠️  {network}: 检测到代理设置")
                    print(f"   HTTP代理: {http_proxy.stdout}")
                    print(f"   HTTPS代理: {https_proxy.stdout}")
    except Exception as e:
        print(f"代理检查失败: {e}")
    
    # 4. 检查 hosts 文件
    print("\n4. 检查 hosts 文件...")
    try:
        with open("/etc/hosts", "r") as f:
            hosts_content = f.read()
            if "google.com" in hosts_content or "play.google.com" in hosts_content:
                print("⚠️  hosts文件中找到Google相关条目")
                # 显示相关行
                for line in hosts_content.split('\n'):
                    if "google" in line:
                        print(f"   {line}")
            else:
                print("✅ hosts文件正常")
    except Exception as e:
        print(f"hosts文件检查失败: {e}")

def test_different_ports():
    """测试不同端口的连接"""
    print("\n5. 测试不同端口连接...")
    
    test_cases = [
        ("http://www.google.com:80", "HTTP (80)"),
        ("https://www.google.com:443", "HTTPS (443)"),
        ("http://www.google.com:8080", "HTTP (8080)"),
    ]
    
    for url, description in test_cases:
        try:
            response = requests.get(url, timeout=10, verify=False)
            print(f"✅ {description}: HTTP {response.status_code}")
        except Exception as e:
            print(f"❌ {description}: {e}")

if __name__ == "__main__":
    deep_network_diagnosis()
    test_different_ports()

=== 深度网络诊断 ===

1. 路由跟踪到 Google...
路由跟踪失败: Command '['traceroute', '-m', '10', 'www.google.com']' timed out after 30 seconds

2. 网络接口状态...
	status: inactive
	status: inactive
	status: inactive
	status: inactive
	status: inactive
	status: inactive
	status: inactive
	status: inactive
	status: active
	status: active
	status: inactive

3. 系统代理设置...
⚠️  HUAWEI_MOBILE: 检测到代理设置
   HTTP代理: Enabled: Yes
Server: 127.0.0.1
Port: 1087
Authenticated Proxy Enabled: 0

   HTTPS代理: Enabled: Yes
Server: 127.0.0.1
Port: 1087
Authenticated Proxy Enabled: 0

⚠️  Thunderbolt Bridge: 检测到代理设置
   HTTP代理: Enabled: Yes
Server: 127.0.0.1
Port: 1087
Authenticated Proxy Enabled: 0

   HTTPS代理: Enabled: Yes
Server: 127.0.0.1
Port: 1087
Authenticated Proxy Enabled: 0

⚠️  Wi-Fi: 检测到代理设置
   HTTP代理: Enabled: Yes
Server: 127.0.0.1
Port: 1087
Authenticated Proxy Enabled: 0

   HTTPS代理: Enabled: Yes
Server: 127.0.0.1
Port: 1087
Authenticated Proxy Enabled: 0


4. 检查 hosts 文件...
✅ hosts文件正常

5. 测试不同端口连接...
✅ HTTP (80): HT

In [8]:
import subprocess
import time

def disable_all_proxies():
    print("=== 禁用所有代理设置 ===")
    
    # 获取所有网络服务
    try:
        result = subprocess.run(["networksetup", "-listallnetworkservices"], 
                              capture_output=True, text=True, timeout=10)
        networks = [line.strip() for line in result.stdout.split('\n') if line.strip() and not line.startswith('*')]
        
        print(f"找到 {len(networks)} 个网络服务")
        
        for network in networks:
            print(f"\n处理网络服务: {network}")
            
            # 禁用HTTP代理
            subprocess.run(["networksetup", "-setwebproxystate", network, "off"], timeout=5)
            print(f"  ✅ 禁用HTTP代理")
            
            # 禁用HTTPS代理
            subprocess.run(["networksetup", "-setsecurewebproxystate", network, "off"], timeout=5)
            print(f"  ✅ 禁用HTTPS代理")
            
            # 禁用SOCKS代理
            subprocess.run(["networksetup", "-setsocksfirewallproxystate", network, "off"], timeout=5)
            print(f"  ✅ 禁用SOCKS代理")
            
            # 清除自动代理配置
            subprocess.run(["networksetup", "-setautoproxyurl", network, ""], timeout=5)
            subprocess.run(["networksetup", "-setautoproxystate", network, "off"], timeout=5)
            print(f"  ✅ 清除自动代理配置")
            
            time.sleep(0.5)
        
        print("\n✅ 所有代理设置已禁用")
        
    except Exception as e:
        print(f"❌ 禁用代理时出错: {e}")

def verify_proxy_disabled():
    """验证代理是否已禁用"""
    print("\n=== 验证代理设置 ===")
    
    try:
        result = subprocess.run(["networksetup", "-listallnetworkservices"], 
                              capture_output=True, text=True, timeout=10)
        networks = [line.strip() for line in result.stdout.split('\n') if line.strip() and not line.startswith('*')]
        
        proxy_found = False
        for network in networks:
            http_proxy = subprocess.run(["networksetup", "-getwebproxy", network], 
                                      capture_output=True, text=True)
            https_proxy = subprocess.run(["networksetup", "-getsecurewebproxy", network], 
                                       capture_output=True, text=True)
            
            if "Yes" in http_proxy.stdout or "Yes" in https_proxy.stdout:
                print(f"❌ {network}: 仍有代理设置")
                proxy_found = True
            else:
                print(f"✅ {network}: 无代理设置")
        
        if not proxy_found:
            print("\n🎉 所有代理设置已成功禁用！")
        else:
            print("\n⚠️  仍有部分代理设置未清除")
            
    except Exception as e:
        print(f"验证失败: {e}")

def test_connectivity_after_fix():
    """修复后测试连接性"""
    print("\n=== 修复后连接测试 ===")
    
    import requests
    import urllib3
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
    
    test_urls = [
        "https://www.google.com",
        "https://play.google.com"
    ]
    
    for url in test_urls:
        try:
            # 使用不验证SSL的方式测试
            response = requests.get(url, timeout=10, verify=False)
            print(f"✅ {url}: HTTP {response.status_code}")
        except Exception as e:
            print(f"❌ {url}: {e}")

if __name__ == "__main__":
    disable_all_proxies()
    verify_proxy_disabled()
    test_connectivity_after_fix()

=== 禁用所有代理设置 ===
找到 4 个网络服务

处理网络服务: An asterisk (*) denotes that a network service is disabled.
** Error: Unable to find item in network database.
  ✅ 禁用HTTP代理
** Error: Unable to find item in network database.
  ✅ 禁用HTTPS代理
** Error: Unable to find item in network database.
  ✅ 禁用SOCKS代理
** Error: The parameters were not valid.
** Error: The parameters were not valid.
  ✅ 清除自动代理配置

处理网络服务: HUAWEI_MOBILE
  ✅ 禁用HTTP代理
  ✅ 禁用HTTPS代理
  ✅ 禁用SOCKS代理
** Error: The parameters were not valid.
  ✅ 清除自动代理配置

处理网络服务: Thunderbolt Bridge
  ✅ 禁用HTTP代理
  ✅ 禁用HTTPS代理
  ✅ 禁用SOCKS代理
** Error: The parameters were not valid.
  ✅ 清除自动代理配置

处理网络服务: Wi-Fi
  ✅ 禁用HTTP代理
  ✅ 禁用HTTPS代理
  ✅ 禁用SOCKS代理
** Error: The parameters were not valid.
  ✅ 清除自动代理配置

✅ 所有代理设置已禁用

=== 验证代理设置 ===
✅ An asterisk (*) denotes that a network service is disabled.: 无代理设置
✅ HUAWEI_MOBILE: 无代理设置
✅ Thunderbolt Bridge: 无代理设置
✅ Wi-Fi: 无代理设置

🎉 所有代理设置已成功禁用！

=== 修复后连接测试 ===
❌ https://www.google.com: HTTPSConnectionPool(host='www.google.com

In [ ]:
from google_play_scraper import reviews, Sort

def test_collection():
    print("Testing review collection...")
    
    try:
        result, _ = reviews(
            "com.openai.chatgpt",
            lang="en",
            country="us", 
            sort=Sort.NEWEST,
            count=10
        )
        
        print(f"Collected {len(result)} reviews")
        for i, review in enumerate(result[:3]):  # 显示前3条
            print(f"{i+1}. {review['content'][:100]}... | Score: {review['score']} | Date: {review['at']}")
            
    except Exception as e:
        print(f"Error: {e}")

if __name__ == "__main__":
    test_collection()

Testing review collection...


In [ ]:
import socket
import subprocess
import requests
import os

def diagnose_network():
    print("=== 网络连接诊断 ===")
    
    # 测试1: 基本网络连通性
    print("\n1. 测试基本网络连通性...")
    test_hosts = [
        "google.com",
        "8.8.8.8",  # Google DNS
        "1.1.1.1"   # Cloudflare DNS
    ]
    
    for host in test_hosts:
        try:
            socket.create_connection((host, 80), timeout=5)
            print(f"✅ {host}: 连接成功")
        except socket.error as e:
            print(f"❌ {host}: 连接失败 - {e}")
    
    # 测试2: DNS解析
    print("\n2. 测试DNS解析...")
    try:
        google_ip = socket.gethostbyname("google.com")
        print(f"✅ DNS解析: google.com -> {google_ip}")
    except socket.gaierror as e:
        print(f"❌ DNS解析失败: {e}")
    
    # 测试3: 系统网络状态
    print("\n3. 系统网络状态...")
    try:
        # macOS 网络诊断命令
        if os.name == 'posix':
            result = subprocess.run(["netstat", "-rn"], capture_output=True, text=True, timeout=10)
            if result.returncode == 0:
                print("✅ 网络路由表正常")
            else:
                print("❌ 网络路由表异常")
    except Exception as e:
        print(f"网络状态检查失败: {e}")
    
    # 测试4: 防火墙状态
    print("\n4. 防火墙状态...")
    try:
        if os.name == 'posix':
            result = subprocess.run(["sudo", "pfctl", "-s", "info"], capture_output=True, text=True, timeout=10)
            if "Status: Enabled" in result.stdout:
                print("⚠️  防火墙已启用 (可能阻止连接)")
            else:
                print("✅ 防火墙状态正常")
    except Exception as e:
        print(f"防火墙检查失败: {e}")

def test_python_requests():
    print("\n5. Python requests 库测试...")
    test_urls = [
        "http://httpbin.org/ip",
        "https://www.google.com",
        "https://play.google.com"
    ]
    
    for url in test_urls:
        try:
            response = requests.get(url, timeout=10)
            print(f"✅ {url}: HTTP {response.status_code}")
        except requests.exceptions.RequestException as e:
            print(f"❌ {url}: {e}")

if __name__ == "__main__":
    diagnose_network()
    test_python_requests()